Test Deployment and run Streamlit App

Test the complete lakehouse agent system end-to-end.

## Prerequisites

- ✅ Run `05-deploy-agent.ipynb` first
- ✅ All components deployed

## What This Notebook Does

1. Tests OAuth token generation from Cognito
2. Tests agent invocation with bearer token
3. Validates end-to-end flow (User → Agent → Gateway → MCP)
4. Verifies agent responses with conversational AI
5. Launches Streamlit UI for interactive testing

## Important Notes

⚠️ **Run cells in order**: Start with the Setup cell (cell 2) to initialize AWS session and clients before running other cells.

In [33]:
# ============================================================================
# SETUP CELL - Run this first to initialize AWS session and clients
# ============================================================================

import sys
sys.path.insert(0, '.')  # Add current directory to path
from pathlib import Path
from aws_session_utils import get_aws_session
import json
import base64
import requests
import uuid
import urllib.parse

# Get validated AWS session with SSO support
session, region, account_id = get_aws_session()

# Initialize AWS clients
ssm_client = session.client('ssm', region_name=region)

print('✅ Setup complete')
print(f'   Region: {region}')
print(f'   Account ID: {account_id}')
print('\n📝 Architecture: User → Agent Runtime → Gateway → MCP Server')

🔍 Using default AWS credentials (no profile specified)
⚠️  No AWS region configured, using default: us-east-1
   To set your region:
   - Environment variable: export AWS_DEFAULT_REGION=your-region
   - AWS CLI: aws configure set region your-region

✅ AWS Credentials Validated
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Profile: default
   Auth method: AWS SSO

✅ Setup complete
   Region: us-east-1
   Account ID: XXXXXXXXXXXX

📝 Architecture: User → Agent Runtime → Gateway → MCP Server


## Step 1: Get OAuth Token from Cognito

In [34]:
import base64  # Import here for cell independence
import requests
import json

# Get Cognito configuration from SSM
COGNITO_DOMAIN = ssm_client.get_parameter(
    Name='/app/lakehouse-agent/cognito-domain'
)['Parameter']['Value']

CLIENT_ID = ssm_client.get_parameter(
    Name='/app/lakehouse-agent/cognito-app-client-id'
)['Parameter']['Value']

CLIENT_SECRET = ssm_client.get_parameter(
    Name='/app/lakehouse-agent/cognito-app-client-secret',
    WithDecryption=True
)['Parameter']['Value']

print(f'🔐 Cognito Configuration:')
print(f'   Domain: {COGNITO_DOMAIN}')
print(f'   Client ID: {CLIENT_ID}')

# Request token
token_url = f'{COGNITO_DOMAIN}/oauth2/token'
credentials = f'{CLIENT_ID}:{CLIENT_SECRET}'
encoded_credentials = base64.b64encode(credentials.encode()).decode()

headers = {
    'Authorization': f'Basic {encoded_credentials}',
    'Content-Type': 'application/x-www-form-urlencoded'
}

data = {
    'grant_type': 'client_credentials',
    'scope': 'lakehouse-api/claims.query'
}

print('\n🔑 Requesting OAuth token...')
response = requests.post(token_url, headers=headers, data=data)

if response.status_code == 200:
    token_data = response.json()
    ACCESS_TOKEN = token_data['access_token']
    print('✅ OAuth token obtained successfully!')
    print(f'   Token type: {token_data.get("token_type")}')
    print(f'   Expires in: {token_data.get("expires_in")} seconds')
else:
    print(f'❌ Failed to get token: {response.status_code}')
    print(response.text)
    ACCESS_TOKEN = None

🔐 Cognito Configuration:
   Domain: https://lakehouse-useast13.auth.us-east-1.amazoncognito.com
   Client ID: comfh80kmr0a3mlrusl6cmulu

🔑 Requesting OAuth token...
✅ OAuth token obtained successfully!
   Token type: Bearer
   Expires in: 3600 seconds


## Step 2: Test Agent Invocation

**Architecture Flow:**
1. User → Agent Runtime (OAuth token in Authorization header for JWT validation)
2. Agent receives token in payload (JWT authorizer consumes header, doesn't pass through)
3. Agent → Gateway (passes token from payload)
4. Gateway → MCP Server (with user context)

**Note:** The bearer token must be passed in BOTH the Authorization header (for JWT validation) AND the payload (for the agent code to use when calling Gateway). This is because the JWT authorizer consumes the Authorization header and doesn't pass it through to the agent code.

In [35]:
import urllib.parse  # Import here for cell independence
import uuid
import json
import requests

if ACCESS_TOKEN:
    # Get Agent Runtime ARN from SSM
    try:
        AGENT_RUNTIME_ARN = ssm_client.get_parameter(
            Name='/app/lakehouse-agent/agent-runtime-arn'
        )['Parameter']['Value']
        
        print(f'🤖 Agent Runtime Configuration:')
        print(f'   Runtime ARN: {AGENT_RUNTIME_ARN}')
        print(f'   Region: {region}')
    except ssm_client.exceptions.ParameterNotFound:
        print('❌ Agent Runtime ARN not found in SSM')
        print('   Please run 05-deploy-agent.ipynb first')
        AGENT_RUNTIME_ARN = None
    
    if AGENT_RUNTIME_ARN:
        # Construct the AgentCore Runtime invocation URL
        # URL encode the agent ARN
        escaped_agent_arn = urllib.parse.quote(AGENT_RUNTIME_ARN, safe='')
        AGENT_RUNTIME_URL = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"
        
        print(f'   Runtime URL: {AGENT_RUNTIME_URL}')
        
        # Generate session ID for this invocation
        session_id = f"test-session-{uuid.uuid4()}"
        
        # Prepare payload with bearer token for Gateway calls
        # Note: Token must be in BOTH header (for JWT auth) and payload (for agent to use)
        payload = {
            'prompt': 'Show me all my claims',
            'bearer_token': ACCESS_TOKEN  # Pass token in payload for agent to use with Gateway
        }
        
        # Prepare headers with OAuth token and session ID
        headers = {
            "Authorization": f"Bearer {ACCESS_TOKEN}",
            "Content-Type": "application/json",
            "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id
        }
        
        print(f'\n🚀 Invoking Agent Runtime...')
        print(f'   Prompt: {payload["prompt"]}')
        print(f'   Session ID: {session_id}')
        print(f'   Auth: Bearer token in header (for JWT validation) and payload (for Gateway)')
        
        try:
            # Call the agent runtime
            response = requests.post(
                AGENT_RUNTIME_URL,
                headers=headers,
                data=json.dumps(payload),
                timeout=60
            )
            
            print(f'\n📊 Response Status: {response.status_code}')
            
            if response.status_code == 200:
                try:
                    result = response.json()
                    print(f'\n✅ Agent Response:')
                    print(json.dumps(result, indent=2))
                    
                    # Display the content if available
                    if 'content' in result:
                        print(f'\n📝 Agent Output:')
                        print(result['content'])
                        
                    if 'tool_calls' in result:
                        print(f'\n🔧 Tool Calls: {result["tool_calls"]}')
                        
                except json.JSONDecodeError:
                    print(f'Response: {response.text[:500]}')
                    
            elif response.status_code == 401:
                print(f'❌ Unauthorized - OAuth token validation failed')
                print(f'   Check that:')
                print(f'      1. Agent Runtime has JWT authorizer configured')
                print(f'      2. Client ID matches the allowed clients')
                print(f'      3. Token has not expired')
                print(f'\n   Response: {response.text[:500]}')
                
            elif response.status_code == 403:
                print(f'❌ Forbidden - User not authorized')
                print(f'   Response: {response.text[:500]}')
                
            elif response.status_code == 424:
                print(f'❌ Failed Dependency - Runtime returned 500 error')
                print(f'   Response: {response.text[:500]}')
                print(f'\n   This means the agent code is crashing.')
                print(f'   Common causes:')
                print(f'      1. Missing Gateway ARN in SSM (/app/lakehouse-agent/gateway-arn)')
                print(f'      2. Agent runtime IAM role lacks SSM permissions')
                print(f'      3. Agent runtime IAM role lacks bedrock-agentcore-control:GetGateway permission')
                print(f'      4. Bearer token not being passed correctly')
                print(f'\n   👉 Run the cells below to diagnose:')
                print(f'      - "Verify Configuration" cell to check SSM parameters')
                print(f'      - "Check CloudWatch Logs" cell to see agent error logs')
                
            else:
                print(f'❌ Request failed')
                print(f'   Response: {response.text[:500]}')
                
        except requests.exceptions.Timeout:
            print(f'\n❌ Request timed out after 60 seconds')
            print(f'   Check CloudWatch logs:')
            print(f'      - Agent Runtime: /aws/bedrock-agentcore/runtime/{AGENT_RUNTIME_ARN.split("/")[-1]}')
            print(f'      - Gateway Interceptor: /aws/lambda/lakehouse-gateway-interceptor')
            
        except Exception as e:
            print(f'\n❌ Error: {e}')
            import traceback
            traceback.print_exc()
else:
    print('⚠️  Skipping agent test - no access token')

🤖 Agent Runtime Configuration:
   Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/lakehouse_agent-HBEJQxHyGT
   Region: us-east-1
   Runtime URL: https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A146666888814%3Aruntime%2Flakehouse_agent-HBEJQxHyGT/invocations?qualifier=DEFAULT

🚀 Invoking Agent Runtime...
   Prompt: Show me all my claims
   Session ID: test-session-7ce1f9cf-7528-4f4e-8195-f96f83369fe3
   Auth: Bearer token in header (for JWT validation) and payload (for Gateway)

📊 Response Status: 200

✅ Agent Response:
{
  "content": "I apologize, but I'm experiencing a technical issue accessing your claims data at the moment. The system is returning an internal error. \n\nCould you please try again in a few moments? If the issue persists, you may want to contact technical support for assistance.\n\nIn the meantime, is there anything else I can help you with, or would you like me to try retrieving a summary of

### Troubleshooting: Check CloudWatch Logs

If you get a 424 error ("Received error (500) from runtime"), the agent code is failing. Let's check the logs.

In [32]:
# Verify all required SSM parameters are set
print('🔍 Verifying SSM Parameter Store Configuration\n')

required_params = {
    '/app/lakehouse-agent/gateway-arn': 'Gateway ARN (required by agent)',
    '/app/lakehouse-agent/gateway-url': 'Gateway URL',
    '/app/lakehouse-agent/agent-runtime-arn': 'Agent Runtime ARN',
    '/app/lakehouse-agent/mcp-server-arn': 'MCP Server ARN',
    '/app/lakehouse-agent/cognito-domain': 'Cognito Domain',
    '/app/lakehouse-agent/cognito-app-client-id': 'Cognito Client ID',
}

missing_params = []
for param_name, description in required_params.items():
    try:
        value = ssm_client.get_parameter(Name=param_name)['Parameter']['Value']
        # Truncate long values for display
        display_value = value if len(value) < 80 else f'{value[:77]}...'
        print(f'✅ {description}')
        print(f'   {param_name}')
        print(f'   Value: {display_value}\n')
    except ssm_client.exceptions.ParameterNotFound:
        print(f'❌ {description}')
        print(f'   {param_name}')
        print(f'   Status: NOT FOUND\n')
        missing_params.append((param_name, description))

if missing_params:
    print(f'\n⚠️  Missing {len(missing_params)} required parameter(s)')
    print('\nThe agent code requires /app/lakehouse-agent/gateway-arn to be set.')
    print('This is the most common cause of 500 errors.')
    print('\nTo fix:')
    for param_name, description in missing_params:
        print(f'   1. Check if {description} was deployed correctly')
        print(f'   2. Verify the deployment script saved to {param_name}')
else:
    print('✅ All required parameters are configured!')

🔍 Verifying SSM Parameter Store Configuration

✅ Gateway ARN (required by agent)
   /app/lakehouse-agent/gateway-arn
   Value: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:gateway/lakehouse-gateway-1e...

✅ Gateway URL
   /app/lakehouse-agent/gateway-url
   Value: https://lakehouse-gateway-1ekjecoowq.gateway.bedrock-agentcore.us-east-1.amaz...

✅ Agent Runtime ARN
   /app/lakehouse-agent/agent-runtime-arn
   Value: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/lakehouse_agent-HBEJ...

❌ MCP Server ARN
   /app/lakehouse-agent/mcp-server-arn
   Status: NOT FOUND

✅ Cognito Domain
   /app/lakehouse-agent/cognito-domain
   Value: https://lakehouse-useast13.auth.us-east-1.amazoncognito.com

✅ Cognito Client ID
   /app/lakehouse-agent/cognito-app-client-id
   Value: comfh80kmr0a3mlrusl6cmulu


⚠️  Missing 1 required parameter(s)

The agent code requires /app/lakehouse-agent/gateway-arn to be set.
This is the most common cause of 500 errors.

To fix:
   1. Check if MCP Server A

### Troubleshooting: Verify Configuration

If the agent returns a 500 error, check that all required SSM parameters are configured.

In [25]:
# Check CloudWatch logs for the agent runtime to diagnose 500 errors
import time

if 'AGENT_RUNTIME_ARN' in locals() and AGENT_RUNTIME_ARN:
    runtime_id = AGENT_RUNTIME_ARN.split('/')[-1]
    log_group_name = f'/aws/bedrock-agentcore/runtime/{runtime_id}'
    
    print(f'🔍 Checking CloudWatch Logs for Agent Runtime')
    print(f'   Log Group: {log_group_name}')
    print(f'   Runtime ID: {runtime_id}\n')
    
    try:
        logs_client = session.client('logs', region_name=region)
        
        # Get recent log streams (last 5 minutes)
        response = logs_client.describe_log_streams(
            logGroupName=log_group_name,
            orderBy='LastEventTime',
            descending=True,
            limit=3
        )
        
        if response['logStreams']:
            print(f'📊 Recent Log Streams ({len(response["logStreams"])}):')
            
            for stream in response['logStreams'][:2]:  # Show last 2 streams
                stream_name = stream['logStreamName']
                print(f'\n   Stream: {stream_name}')
                
                # Get recent log events
                try:
                    events_response = logs_client.get_log_events(
                        logGroupName=log_group_name,
                        logStreamName=stream_name,
                        limit=20,
                        startFromHead=False
                    )
                    
                    if events_response['events']:
                        print(f'   Last {len(events_response["events"])} log entries:\n')
                        for event in events_response['events'][-10:]:  # Show last 10
                            timestamp = time.strftime('%Y-%m-%d %H:%M:%S', 
                                                     time.localtime(event['timestamp']/1000))
                            message = event['message'].strip()
                            print(f'   [{timestamp}] {message}')
                    else:
                        print(f'   No recent events')
                        
                except Exception as e:
                    print(f'   Error reading events: {e}')
        else:
            print('❌ No log streams found')
            print('   The agent may not have been invoked yet')
            
    except logs_client.exceptions.ResourceNotFoundException:
        print(f'❌ Log group not found: {log_group_name}')
        print('   The agent may not have been invoked yet, or logging is not configured')
    except Exception as e:
        print(f'❌ Error checking logs: {e}')
        
    print(f'\n💡 To view logs in real-time:')
    print(f'   aws logs tail {log_group_name} --follow')
else:
    print('⚠️  Agent Runtime ARN not available. Run the agent invocation cell first.')

🔍 Checking CloudWatch Logs for Agent Runtime
   Log Group: /aws/bedrock-agentcore/runtime/lakehouse_agent-HBEJQxHyGT
   Runtime ID: lakehouse_agent-HBEJQxHyGT

❌ Log group not found: /aws/bedrock-agentcore/runtime/lakehouse_agent-HBEJQxHyGT
   The agent may not have been invoked yet, or logging is not configured

💡 To view logs in real-time:
   aws logs tail /aws/bedrock-agentcore/runtime/lakehouse_agent-HBEJQxHyGT --follow


## Step 3: Verify User Context

Test that user identity is propagated correctly.

In [ ]:
# This would require testing with different user tokens
print('📋 User Context Verification:')
print('   To fully test user context propagation:')
print('   1. Get tokens for different users (user001, user002)')
print('   2. Query claims with each token')
print('   3. Verify Gateway interceptor extracts user identity')
print('   4. Check X-User-Principal header is added')
print('\n   Use the Streamlit UI for interactive testing!')
print('   Or check CloudWatch logs for user identity propagation.')

## Step 4: Check CloudWatch Logs

In [ ]:
print('📊 CloudWatch Logs to Check:')
print('\n1. Interceptor Lambda:')
print('   aws logs tail /aws/lambda/lakehouse-gateway-interceptor --follow')
print('\n2. MCP Server:')
print('   Check AgentCore Runtime logs in CloudWatch')
print('\n3. Look for:')
print('   ✅ Bearer token extracted from MCP gateway request')
print('   ✅ Extracted user principal: user@example.com')
print('   ✅ Request authorized for user: user@example.com')

## Step 5: Launch Streamlit UI

Launch the interactive Streamlit UI for conversational testing with the agent.

In [ ]:
import subprocess
import os

print('🚀 Launching Streamlit UI...')
print('\n📝 Instructions:')
print('   - Streamlit will open in your browser automatically')
print('   - Login with: user001@example.com / TempPass123!')
print('   - Try queries like: "Show me all claims" or "Get claims summary"')
print('   - Press Ctrl+C in the terminal to stop Streamlit')
print('\n⏳ Starting Streamlit server...')

# Change to streamlit-ui directory and run streamlit
try:
    streamlit_dir = os.path.join(os.getcwd(), 'streamlit-ui')
    subprocess.run(
        ['streamlit', 'run', 'streamlit_app.py'],
        cwd=streamlit_dir,
        check=True
    )
except KeyboardInterrupt:
    print('\n\n✅ Streamlit stopped')
except FileNotFoundError:
    print('\n❌ streamlit-ui directory or streamlit_app.py not found')
    print('   Make sure you are running this from the lakehouse-agent directory')
except Exception as e:
    print(f'\n❌ Error launching Streamlit: {e}')
    print('\n💡 Manual launch:')
    print('   cd streamlit-ui')
    print('   streamlit run streamlit_app.py')

## Summary

✅ **Testing Complete!**

**Architecture Validated:**
```
User (with OAuth token)
  ↓
AgentCore Runtime (Lakehouse Agent)
  ├─ Validates user OAuth token (JWT authorizer)
  ├─ Extracts token from Authorization header
  ↓
AgentCore Gateway
  ├─ Receives bearer token from agent
  ├─ Interceptor Lambda validates token
  ├─ Adds user identity (X-User-Principal header)
  ↓
MCP Athena Server
  └─ Executes queries with user context
```

**What was tested:**
- User OAuth token generation from Cognito
- Agent runtime invocation with bearer token in header
- Agent → Gateway → MCP Server flow
- User identity propagation through headers
- Interactive Streamlit UI for conversational testing

**Additional Testing:**

1. **Test with different users:**
   - user001@example.com / TempPass123!
   - user002@example.com / TempPass123!

2. **Verify User Context:**
   - Check that Gateway interceptor extracts user identity
   - Verify X-User-Principal header is added to MCP requests
   - Confirm user identity appears in CloudWatch logs

**Troubleshooting:**
- Check CloudWatch logs for:
  - Agent Runtime logs: `/aws/bedrock-agentcore/runtime/<runtime-id>`
  - Gateway Interceptor logs: `/aws/lambda/lakehouse-gateway-interceptor`
  - Look for: "Bearer token extracted", "User: <email>", "Request authorized"
- Verify SSM parameters are set correctly
- Ensure all components are deployed in correct order